# 02 · T5 — Training the three models

**Deliverable (plan, section 4):** `models/<dataset>/seed<k>/*` and an accuracy CSV (`reports/accuracies.csv`).

| kind | model | used by |
|---|---|---|
| `lr` | logistic regression in **PyTorch** | every method (Wachter included) |
| `lr_sk` | logistic regression, sklearn | check only (T5 criterion) |
| `rf` | random forest | model-agnostic methods only |
| `nn` | PyTorch MLP, 2 hidden layers (32, 16), ReLU | every method (Wachter included) |

All of them take the **same vector** as the rest of `cfqd` (YAML order, categorical codes 0..k-1) and expose
`predict_proba`; the encoding (standardization, one-hot) lives inside the model. One seed controls the split (70/30
stratified), the training and the search. Hyperparameters are fixed (`models.DEFAULTS`) and were **not** tuned on the
test set.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)

In [2]:
from cfqd import datasets as D, models as M

DATASETS = ["german_credit", "adult", "heloc", "gmc", "compas_guidotti", "simulation"]
rows, trained = [], {}
for n in DATASETS:
    ds = D.load_dataset(n, seed=0)
    ms = M.train_all(ds)  # loads models/<n>/seed0 if it exists
    trained[n] = (ds, ms)
    for k, m in ms.items():
        rows.append(dict(dataset=n, model=k, **M.evaluate(m, ds)))
acc = pd.DataFrame(rows)
acc.pivot(index="dataset", columns="model", values="acc_test").round(3)

model,lr,lr_sk,nn,rf
dataset,,,,
adult,0.849,0.849,0.857,0.865
compas_guidotti,0.672,0.673,0.673,0.675
german_credit,0.733,0.733,0.713,0.717
gmc,0.935,0.935,0.934,0.935
heloc,0.717,0.718,0.722,0.724
simulation,0.894,0.894,0.961,0.971


## 1. Check against the literature (Guidotti, 70/30 split)

Compare as a **range** (±0.05), not as equality. Caveats from the plan: Guidotti's HELOC is the raw file
(10,459 rows), not OpenML 45554; and we do not know whether his `german` is the old Statlog or the corrected one.

In [3]:
ref = []
for n, d in M.GUIDOTTI_ACC.items():
    if n not in trained:
        continue
    for k, v in d.items():
        ours = float(acc.query("dataset == @n and model == @k")["acc_test"].iloc[0])
        ref.append(dict(dataset=n, model=k, guidotti=v, ours=round(ours, 3), diff=round(ours - v, 3),
                        within_005=abs(ours - v) <= 0.05))
ref = pd.DataFrame(ref)
ref

,dataset,model,guidotti,ours,diff,within_005
0,adult,rf,0.85,0.865,0.015,True
1,adult,nn,0.84,0.857,0.017,True
2,german_credit,rf,0.76,0.717,-0.043,True
3,german_credit,nn,0.81,0.713,-0.097,False
4,heloc,rf,0.68,0.724,0.044,True
5,heloc,nn,0.67,0.722,0.052,False
6,compas_guidotti,rf,0.56,0.675,0.115,False
7,compas_guidotti,nn,0.61,0.673,0.063,False


### German Credit: variance across seeds

The German test set has only 300 rows: one seed is not enough to judge the range.

In [4]:
g = []
for s in range(5):
    ds = D.load_dataset("german_credit", seed=s)
    for k in ["lr", "rf", "nn"]:
        g.append(dict(seed=s, model=k, acc=M.evaluate(M.get_model(ds, k), ds)["acc_test"]))
g = pd.DataFrame(g)
g.groupby("model")["acc"].agg(["mean", "std", "min", "max"]).round(3)

,mean,std,min,max
model,,,,
lr,0.753,0.022,0.733,0.780
nn,0.737,0.022,0.713,0.767
rf,0.747,0.021,0.717,0.767


**Reading:** averaged over 5 seeds the RF is within Guidotti's range (0.76). The neural network is ~0.07 below the
published 0.81. Typical MLP values for German Credit in the literature are 0.74–0.77, and 0.81 cannot be reproduced
without his architecture. We did not tune hyperparameters on the test set. **Recorded as a known difference.**

## 2. COMPAS: label leakage?

The plan says "COMPAS above 0.65 signals label leakage". Our accuracy is ~0.67. Check:

In [5]:
ds_c, ms_c = trained["compas_guidotti"]
print("inputs:", ds_c.spec.names)
print("label-leaking columns among the inputs:", sorted(set(ds_c.spec.names) & D.COMPAS_LEAKY) or "none")
print("majority-class baseline:", round(M.evaluate(ms_c["lr"], ds_c)["majority_baseline"], 3))

inputs: ['age', 'sex', 'race', 'c_charge_degree', 'priors_count', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'length_of_stay']
label-leaking columns among the inputs: none
majority-class baseline: 0.545


**Reading:** no post-assessment column (`is_recid`, `decile_score`, `score_text`, `r_*`, `vr_*`, ...) enters the
model: a guard in `datasets._prep_compas` fails if that ever happens. ~0.67 is exactly what Dressel & Farid (2018,
*Science Advances*) obtain with logistic regression on age, sex, priors and charge degree (≈0.67). Guidotti's
0.56/0.61 use another target or another feature set. **The plan's 0.65 threshold seems miscalibrated for this
target (`two_year_recid`); we propose replacing it by the explicit column check above.**

## 3. Logistic regression: PyTorch vs sklearn

In [6]:
lr_cmp = []
for n, (ds, ms) in trained.items():
    a, b = ms["lr"], ms["lr_sk"]
    lr_cmp.append(dict(dataset=n, acc_torch=M.evaluate(a, ds)["acc_test"], acc_sklearn=M.evaluate(b, ds)["acc_test"],
                       prediction_agreement=float(np.mean(a.predict(ds.X_test) == b.predict(ds.X_test))),
                       max_proba_diff=float(np.abs(a.predict_proba(ds.X_test) - b.predict_proba(ds.X_test)).max())))
lr_cmp = pd.DataFrame(lr_cmp)
lr_cmp["acc_diff"] = (lr_cmp["acc_torch"] - lr_cmp["acc_sklearn"]).abs()
lr_cmp.round(4)

,dataset,acc_torch,acc_sklearn,prediction_agreement,max_proba_diff,acc_diff
0,german_credit,0.7333,0.7333,1.0000,0.0088,0.0000
1,adult,0.8495,0.8494,0.9992,0.1354,0.0001
2,heloc,0.7171,0.7178,0.9993,0.0044,0.0007
3,gmc,0.9349,0.9349,1.0000,0.0044,0.0000
4,compas_guidotti,0.6722,0.6728,0.9984,0.0516,0.0005
5,simulation,0.8939,0.8939,1.0000,0.0004,0.0000


## 4. The same `predict_proba` call for the three models (through `CountingModel`)

In [7]:
from cfqd import CountingModel
ds, ms = trained["german_credit"]
for k in ["lr", "rf", "nn"]:
    cm = CountingModel(ms[k], budget=100)
    p = cm.predict_proba(ds.X_test[:7])
    print(k, type(ms[k]).__name__, p.shape, "queries counted:", cm.n_queries,
          "| differentiable:", getattr(ms[k], "differentiable", False))

lr TorchModel (7, 2) queries counted: 7 | differentiable: True
rf SklearnModel (7, 2) queries counted: 7 | differentiable: False
nn TorchModel (7, 2) queries counted: 7 | differentiable: True


In [8]:
out = ROOT / "reports"
out.mkdir(exist_ok=True)
acc.to_csv(out / "accuracies.csv", index=False)
print("written:", (out / "accuracies.csv").relative_to(ROOT).as_posix())

written: reports/accuracies.csv


## 5. T5 acceptance checklist (plan 4.3)

In [9]:
checks = {
    "accuracies within ~0.05 of the references (seed 0)": bool(ref["within_005"].all()),
    "COMPAS has no label-leaking column": not (set(ds_c.spec.names) & D.COMPAS_LEAKY),
    "COMPAS not above 0.65 (plan criterion; see the reading above)":
        bool((acc.query("dataset == 'compas_guidotti'")["acc_test"] <= 0.65).all()),
    "LR torch vs sklearn: accuracy difference < 0.01": bool((lr_cmp["acc_diff"] < 0.01).all()),
    "the three models answer the same predict_proba call": True,
}
for k, v in checks.items():
    print("PASS" if v else "FAIL", k)

FAIL accuracies within ~0.05 of the references (seed 0)
PASS COMPAS has no label-leaking column
FAIL COMPAS not above 0.65 (plan criterion; see the reading above)
PASS LR torch vs sklearn: accuracy difference < 0.01
PASS the three models answer the same predict_proba call
